# Topic modeling
En este notebook se va a demostrar el uso de distintos modelos de extracción de temáticas (*topic modeling*) en un conjunto de textos de ejemplo sencillo (sin Deep Learning, para comprender sus limitaciones).

In [1]:
!pip install textacy

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/946.5 kB ? eta -:--:--
   ---------------------------------------- 946.5/946.5 kB 6.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----------------------------------- ---- 1.8/2.1 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 10.0 MB/s  0:00:00
Failed to build floret


  error: subprocess-exited-with-error
  
  × Building wheel for floret (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
      C:\Users\joanm\AppData\Local\Temp\pip-build-env-hena3x_o\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: MIT License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-313\f

In [ ]:
!pip install textacy
!python -m spacy download en_core_web_md

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.6/321.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 15.1 MB/s eta 0:00:00


In [1]:
import spacy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

nlp=spacy.load('es_core_news_md')

### Creación del corpus
Creamos un pequeño Corpus de ejemplo formado por 8 frases cortas. Definimos una sencilla función de normalización y aplicamos esta normalización a todo el corpus.

In [ ]:
import pandas as pd
import spacy

def extract_lemmas(doc):
    """Filtra y lematiza un documento ya procesado por spaCy."""
    return ' '.join(t.lemma_ for t in doc if not (t.is_punct or t.is_space or t.is_stop))

df = pd.read_csv('datos_clean.csv')

norm_corpus = list(map(extract_lemmas, nlp.pipe(df['texto'], batch_size=256,n_process=-1)))

## Topic modeling usando Scikit-learn
La librería `scikit-learn` implementa los modelos *Latent Semantic Analysis* (LSA) y *Latent Dirichlet Allocation* (LDA).  
Partimos de un modelo TF-IDF para el modelado LSA y de un modelo BoW para el modelado LDA

### Modelo LSA

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# usamos características tf-idf para LSA.
tfidf_vectorizer = TfidfVectorizer(min_df=2)
tfidf = tfidf_vectorizer.fit_transform(norm_corpus)

Definimos una función de ayuda para mostrar los resultados (términos asociados a cada tema)

In [7]:
def print_top_words(model, feature_names, n_top_words):
    """Función auxiliar para mostrar los términos más importantes
    de cada topic"""
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic #{topic_idx}: "
        message += " ".join([feature_names[i]
                             for i in topic.argsort()[:-n_top_words - 1:-1]])  # construye el texto con las palabras más importantes del topic
        print(message)
    print()

Calculamos los modelos para nuestro corpus (método `fit`) y vemos cuáles son los 3 términos con más peso para cada *topic*. Cada modelo asigna un grado de pertenencia en cada tema a cada término del vocabulario de la matriz tfidf o bow utilizada como entrada.

In [10]:
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

# Ajustamos el modelo LSA
lsa = TruncatedSVD(n_components=2).fit(tfidf)

print("\nTopics en modelo LSA:")
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
print(tfidf_feature_names)
print_top_words(lsa, tfidf_feature_names, 3)


Topics en modelo LSA:
['dog' 'fox' 'java' 'language' 'programming' 'python']
Topic #0: fox dog java
Topic #1: programming java language



In [11]:
for topic_idx, topic in enumerate(lsa.components_):
    print(f"\nTopic #{topic_idx}")
    for i, weight in enumerate(topic):
        print(f"{tfidf_feature_names[i]}: {weight:.3f}")


Topic #0
dog: 0.707
fox: 0.707
java: 0.000
language: -0.000
programming: -0.000
python: -0.000

Topic #1
dog: -0.000
fox: 0.000
java: 0.500
language: 0.500
programming: 0.500
python: 0.500


El método `fit` aprende la matriz de `topics` x `términos` para el corpus dado

In [12]:
lsa.components_.shape

(2, 6)

In [13]:
pd.DataFrame(np.round(lsa.components_, 4), columns=tfidf_feature_names)

,dog,fox,java,language,programming,python
0,0.7071,0.7071,0.0,-0.0,-0.0,-0.0
1,-0.0000,0.0000,0.5,0.5,0.5,0.5


Podemos ver el porcentaje de pertenencia a cada *topic* de cada una de los documentos asignados por el modelo con el método `transform`:

In [14]:
lsa.transform(tfidf)

array([[ 1.00000000e+00,  1.66533454e-16],
       [ 7.07106781e-01,  2.35513869e-16],
       [ 7.07106781e-01,  0.00000000e+00],
       [ 1.00000000e+00,  1.66533454e-16],
       [-1.92296269e-16,  8.66025404e-01],
       [-1.28197512e-16,  8.66025404e-01],
       [-1.66533454e-16,  1.00000000e+00],
       [-7.85046229e-17,  7.07106781e-01]])

In [16]:
np.round(lsa.transform(tfidf), 4)

array([[ 1.    ,  0.    ],
       [ 0.7071,  0.    ],
       [ 0.7071,  0.    ],
       [ 1.    ,  0.    ],
       [-0.    ,  0.866 ],
       [-0.    ,  0.866 ],
       [-0.    ,  1.    ],
       [-0.    ,  0.7071]])

Cada fila corresponde a un documento del Corpus, y cada columna el grado de pertenencia a ese tema del documento.  
El modelo ha separado correctamente el corpus en las dos temáticas principales:

In [17]:
np.argmax(lsa.transform(tfidf), axis=1)

array([0, 0, 0, 0, 1, 1, 1, 1])

### Modelo LDA

In [19]:
# usamos características BoW para LDA.
tf_vectorizer = CountVectorizer(min_df=2)
tf = tf_vectorizer.fit_transform(norm_corpus)

In [20]:
# Ajustamos el modelo LDA
lda = LatentDirichletAllocation(n_components=2, max_iter=5,
                                learning_method='batch',
                                learning_offset=50.,
                                random_state=0).fit(tf)

print("\nTopics en modelo LDA:")
tf_feature_names = tf_vectorizer.get_feature_names_out()
print_top_words(lda, tf_feature_names, 3)


Topics en modelo LDA:
Topic #0: dog fox java
Topic #1: programming language python



El atributo `components_` contiene los parámetros de la distribución de términos en *topics*.

In [ ]:
lda.components_.shape

(2, 6)

In [21]:
pd.DataFrame(lda.components_, columns=tfidf_feature_names)

,dog,fox,java,language,programming,python
0,3.491959,3.491955,0.513787,0.511393,0.511393,0.513756
1,0.508041,0.508045,3.486213,3.488607,3.488607,3.486244


Normalizando esta matriz muestra la distribución de términos dentro de cada *topic*

In [22]:
distribucion = lda.components_ / lda.components_.sum(axis=1)[:, np.newaxis]
pd.DataFrame(distribucion, columns=tfidf_feature_names)

,dog,fox,java,language,programming,python
0,0.386525,0.386524,0.056871,0.056606,0.056606,0.056868
1,0.033947,0.033947,0.232946,0.233106,0.233106,0.232948


Como en el caso del LSA, podemos ver el grado de pertenencia de cada documento a cada *topic*

In [23]:
lda.transform(tf)

array([[0.8319788 , 0.1680212 ],
       [0.74801655, 0.25198345],
       [0.74801659, 0.25198341],
       [0.8319788 , 0.1680212 ],
       [0.1281347 , 0.8718653 ],
       [0.12813488, 0.87186512],
       [0.1025063 , 0.8974937 ],
       [0.17085046, 0.82914954]])

In [ ]:
np.argmax(lda.transform(tf), axis=1)

array([0, 0, 0, 0, 1, 1, 1, 1])